# Modulo 1: Nivel de aplicacion. Tema 2: Protocolo HTTP

**<u>Informacion del protocolo</u>**  

**Versiones**  

<img src="Versiones.png" style="display: block;">

**Metodos**  

<img src="Metodos.png" style="display: block;">

**Codigos de respuesta** 

<img src="Codigos-respuesta.png" style="display: block;">

## Peticiones y respuestas HTTP

**Formato general de una peticion HTTP**

<img src="Formato-peticion.png" style="display: block; margin: auto;">

Se distinguen tres partes separadas por <strong>\r\n</strong>:
- Peticion (Request Line): <strong>[Operacion] [URI] HTTP/[Version]</strong>
    - Operacion: de la lista de arriba (GET, POST, HEAD, ...)
    - URI: direccion del recurso en la que aplicar la operacion. Tambien se puede utilizar una URL (incluye la direccion de la web al principio)
    - Version: version de HTTP que quieres usar
- Cabeceras (Request Headers): <strong>[Id]: [Valor]</strong>, cada uno separado por un retorno de carro y un salto de linea. Son opcionales (casi todas) y dan informacion adicional sobre el cliente al servidor
    - ID: identificador de la cabecera
    - Valor: valor de esa cabecera
- Cuerpo (Request Message Body): no tiene sintaxis concreta. Da informacion de l apagina web, no del servidor. Se utiliza especialmente en las respuestas del servidor (ya que en el cuerpo se mete el recurso que habias pedido) y en formularios (para indicar al servidor que pone en cada campo)

**Formato de una respuesta HTTP**

<img src="Formato-respuesta.png" style="display: block; margin: auto;">

Estructurado de manera similar:
- Estado (Status Line): <strong>HTTP/[Version] [Codigo] [Texto]</strong>
    - Version: version de HTTP que quieres usar
    - Codigo: de la lista de arriba (200, 404, 500, ...)
    - Texto: texto breve ecxplicando el codigo de respuesta
- Cabeceras (Headers): igual que las peticiones. Existen cabeceras solo para peticiones (como Accept) y solo para respuestas (como Accept-Ranges)
- Cuerpo (body): igual que en las peticiones

## ACT-T1-02 Conectando a un servidor Web
Netcat es una herramienta sencilla para realizar conexiones raw y permite enviar peticiones “a mano”. Nos conectamos al servidor con el siguiente comando: 
```bash  
nc -C -v uam.es 80. 
```
- -v: ‘verbose’, es decir, el comando va a hacer lo mismo, solo que te va a escribir en la terminal mas informacion de la normal.
- 80: puerto al que conectarse
  Enviaremos la peticion HTTP: GET / HTTP/1.1

In [4]:
!echo -e "GET / HTTP/1.1\r\nHost: uam.es\r\nConnection: close\r\n\r\n" | nc -C -v uam.es 80

Connection to uam.es (150.244.214.237) 80 port [tcp/http] succeeded!
HTTP/1.1 307 Moved Temporarily
Location: https://uam.es/
Content-Length: 0



Se puede ver que el servicio ha sido movido de localizacion, ya que ahora no se usa el protocolo http, se usa el https que es mas moderno y seguro

## Conexiones persistentes y no persistentes

<img src="Conexiones.png" style="display: block; margin: auto;">

- **No persistente**: La conexión TCP se establece y se <strong>cierra</strong> para cada recurso que queremos sacar del servidor (html, imagenes, cookie, …). Los navegadores abren multiples conexiones simultaneas. Es la opcion por defecto en HTTP/1.0
- **Persistente**: la conexión <strong>no se cierra</strong>, de forma que pueden enviarse varios objetos por una unica conexión TCP. Es la opcion por defecto en HTTP/1.1. Permite tecnicas como el pipelining.
- **Pipelining**: envio de peticiones aunque aun no se haya recibido la respuesta a la anterior. El servidor debe enviar las respuestas en el orden en el que se reciben, de manera que utiliza una cola FIFO.
Si un paquete tarda en processarase en el servidor, bloquea toda la cola, ralentizando toda la conexión.

En ejercicios y modelos, consideramos un modelo lo mas realista posible del comportamiento de un navegador moderno (Chrome), a no ser que se inidque lo contrario:
- Cargan completamente una pagina Web, y solo despues, solicita los recursos que contiene
- Tiene un limite de 6 conexiones simultaneas con un mismo servidor, y 10 en total
- Si puede reutiliza conexiones abiertas

Ejemplo: Cuando entras a uam.es, el proceso es el siguiente:

<img src="Ejemplo.png" style="display: block; margin: auto;">

1. Conexión TCP con el servidor web de la UAM. Para este proceso se necesitan enviar 3 paquetes (explicación en los apuntes del tema 1). Teniendo en cuenta que RTT (Round-Trip Time) representa la ida y vuelta de un mensaje, se puede ver que este paso requiere 1,5 RTT.
2. Solicitud de la página web (el archivo html que queremos que el servidor nos envíe). Esto requiere mandar una petición al servidor y esperar su respuesta; es decir, 2 paquetes = 1 RTT. Sin embargo, HTTP se aprovecha del paso anterior: la petición la envía combinada con el tercer paquete del emparejamiento TCP.

Es decir, en el caso ideal (es decir, el archivo es pequeño) pedir un recurso nada más entrar a una página web requiere 2 RTT. En las diapositivas pone que 2,5 RTT. Eso está mal. En el libro de referencia (James F. Kurose, Keith W. Ross. (2010) Redes de computadoras. Un enfoque descendente. 7ª edición) lo explica (Figure 2.7). También lo he visto en este paper publicado en w3.org.
Además, si la conexión es persistente, una vez realizada la conexión TCP puedes pedir el resto de recursos sin tener que repetir el paso 1; esto es, solo tardará 1 RTT.

## HTTP/2
- <strong>Compatible</strong> con las versiones HTTP anteriores, por lo que la sintaxis y codigos se mantienen igual
- Una unica conexión TCP
- Objetos con **prioridades** (entre 1 y 256), ya que hay elementos de una pagina que te interesa cargar antes que otros.
- Permite **Server push**, de manera que el servidor “anticipa” que vas a necesitar un recurso y te lo envia aunque no haya recibido aun la peticion
- Los objetos se dividen en **frames** para mitigar el bloqueo HOL (bloqueo que ocurre con FCFS que los objetos pequeños tienen que esperar mucho si estan detras de objetos grandes)

<div style="display: flex; justify-content: center; align-items: flex-start; gap: 0;">
  <div style="text-align: center;">
    <figure>
      <img src="Problema.png" style="width: 100%; max-width: 600px; height: auto; object-fit: contain;">
      <figcaption>Problema</figcaption>
    </figure>
  </div>
  <div style="text-align: center;">
    <figure>
      <img src="Solucion.png" style="width: 100%; max-width: 600px; height: auto; object-fit: contain;">
      <figcaption>Solución</figcaption>
    </figure>
  </div>
</div>

## Manteniendo el estado
HTTP es un protocolo sin estado, es decir, en el servidor cada peticion se trata de manera independiente sin establecer ninguna relacion con las anteriores.  
Una transaccion Web normal necesita “recordar” quienes somos, para ello el servidor proporciona al navegador pequeñas piezas de informacion (cookie) unicas para cada usuario. Este las guarda y las vuelve a entregar al servidor cuando se visita el mismo dominio que genero la cookie.  

**<u>Cookies</u>**  
Hoy en dia, la herramienta mas utilizada para informar al servidor de las peticiones anteriores es la Cookie. Consiste en un identificador unico para tu ordenador en una pagina web. De esta manera, el servidor guarda un registro de todos estos identificadores que existen y asi puede controlar la actividad de cada uno.

**Funcionamiento**:

<img src="Cookies.png" style="display: block; margin: auto;">

1. El cliente consulta si tiene una cookie para amazon.es. Como es la primera vez que se conecta a Amazon, no encuentra nada. Manda una petición HTTP habitual, es decir, sin cookie: ```GET / HTTP/1.1\r\nHost: amazon.es```.
2. El servidor recibe esta petición. Antes de devolver el recurso solicitado (en este caso, la raíz de la página), ve que la petición no tiene la cabecera llamada **Cookie** . El servidor reserva espacio para el nuevo cliente (carrito de la compra, historial de compras, …) y a esta memoria le da el **id 1678** . Este identificador es nuestra cookie.  
El servidor incluye en la respuesta HTTP la cabecera ```Set-cookie: 1678```.
3. El cliente ve la cabecera **Set-cookie** y entiende que el servidor le ha asignado este identificador. De esta manera, en sus próximas peticiones se encargará de realizarlas con la cookie (utilizando la cabecera Cookie: **1678**) para que el servidor pueda buscar en su base de datos, encontrar este id y devolverle contenido especifico para él.  
Por defecto, ***la Cookie será eliminada al cerrar el navegador***. Sin embargo, el servidor puede cambiar su configuración y ponerle **fecha de caducidad**, de manera que sea válida hasta que llegue esa fecha.

**<u>Tokens</u>**  
Las aplicaciones Web necesitan autenticar y autorizar a sus usuarios
- **Autenticacion**: comprobar que un usuario es quien dice ser (tipicamente, con usuario/contraseña)
- **Autorizacion**: comprobar que un usuario atuenticado tiene permisos para realizar cierta accion
Cuando se utilizan para autenticar, las cookies deben comprobarse accediendo a la BD en cada peticion del usuario → muy mal rendimiento. Los tokens no tienen este problema: cada token esta firmado, lo que permite comprobar su validez sin mas informacion. Se envia como una cabecera en cada peticion: ```Authorization: Bearer {Token}```

**JSON Web Token (JWT)**

<img src="JWT.png" style="display: block; margin: auto;">

El string esta compuesto de 3 partes (separadas por un punto en el string):
- **Header**: encabezado donde se indica, al menos, el algoritmo y el tipo de token, que en el caso del ejemplo anterior era el algoritmo HS256 y un token JWT.
- **Payload**: donde aparecen los datos de usuario y privilegios, así como toda la información que queramos añadir, todos los datos que creamos convenientes.
- **Signature**: una firma que nos permite verificar si el token es válido, y aquí es donde radica el quid de la cuestión, ya que si estamos tratando de hacer una comunicación segura entre partes y hemos visto que podemos coger cualquier token y ver su contenido con una herramienta sencilla, ¿dónde reside entonces la potencia de todo esto?  
La firma se construye de tal forma que vamos a poder verificar que el remitente es quien dice ser, y que el mensaje no ha sido modificado por el camino. Se construye como el HMACHSHA256, que son las siglas de Hash-Based Message Authentication Code (Código de Autenticación de Mensajes), cifrado además con el algoritmo SHA de 256 bits. Se aplica esa función a:
    - **Codificación en Base64 de header**.
    - **Codificación en Base64 de payload**.
    - **Un secreto**, establecido por la aplicación.  

De esta forma, si alguien modifica el token por el camino, podriamos verificar que la comprobacion de la firma no es correcta, por lo que no podemos confiar en el token recibido y deberiamos denegar la solicitud de recursoso que nos haya realizado.  
La interaccion cliente-servidor usando JWT seria la siguiente:

<img src="Interaccion-JWT.png" style="display: block; margin: auto;">

## Caches Web

<div style="display: flex; align-items: flex-start; gap: 20px;">
  <!-- Texto a la izquierda -->
  <div style="flex: 1; font-family: sans-serif;">
    <p><strong>Objetivo:</strong> evitar que las peticiones de un cliente tengan que viajar hasta el servidor de destino</p>
    <p>El tráfico del usuario se configura para que viaje a través de un servidor cache Web local:</p>
    <ul>
      <li>si objeto en cache: se devuelve el objeto</li>
      <li>sino la cache pide el recurso al servidor, lo guarda para el futuro y devuelve al cliente</li>
    </ul>
  </div>

  <!-- Imagen a la derecha -->
  <div style="flex: 1; text-align: center;">
    <img src="Cache-Web.png" alt="Esquema caché web" style="max-width: 100%; height: auto; border: 1px solid #ccc;">
  </div>
</div>

Esta solucion reduce los tiempos de respuesta ya que la cache esta mas cerca (a menos saltos). El servidor decide que conteinido puede cachearse, influyendo cabeceras en las respuestas HTTP: ```Cache-Control: max-age=<seconds>``` o ```Cache-Control: no-cache```

<div style="display: flex; align-items: flex-start;">
  <div style="flex: 1; padding-right: 20px;">
    <h3><u>GET condicional</u></h3>
    <p><strong>Objetivo:</strong> no realizar peticiones si el navegador tiene una versión actualizada del objeto solicitado</p>
    <ul>
      <li><strong>Cliente</strong>: indica la fecha de la versión cacheada con cabecera específica en la petición HTTP: 
        <code>If-modified-since: &lt;fecha&gt;</code>
      </li>
      <li><strong>Servidor</strong>: si la copia está actualizada, no se devuelve el objeto: 
        <code>HTTP/1.1 304 Not Modified</code>
      </li>
    </ul>
  </div>
  <div style="flex: 1;">
    <img src="Get-Condicional.png" alt="GET Condicional" style="max-width: 100%;">
  </div>
</div>


<img src="Escenario-real1.png" style="display: block; margin: auto;">

<img src="Escenario-real2.png" style="display: block; margin: auto;">
